<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M12/M12_Workshop2_Google_ADK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">🌐 M12 Workshop 2 — Google ADK</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐⭐⭐ Difficulty: Advanced &nbsp;|&nbsp; ⏱ Time: ~15 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Install and configure the <b>Google Generative AI SDK</b></li>
    <li>Build an agent using <b>Gemini's tool/function calling</b></li>
    <li>Implement a <b>multi-step research agent</b> with Gemini</li>
    <li>Compare <b>Google ADK vs Claude SDK vs OpenAI</b> approaches</li>
    <li>Build a <b>planning agent</b> that breaks tasks into steps</li>
  </ol>
</div>

## 📦 Setup

> **🔑 API Key Setup:** Add `GEMINI_API_KEY` to Colab Secrets. Also keep `OPENAI_API_KEY` for comparison.

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade google-generativeai openai
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
from dads5250 import setup_openai, show_response, show_expected, show_info, quiz

openai_client = setup_openai()

import google.generativeai as genai
from google.colab import userdata
import json

genai.configure(api_key=userdata.get("GEMINI_API_KEY"))
print(f"✅ Google Generative AI SDK ready")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ Gemini Function Calling</h2>
</div>

Google's approach to tool use is called **Function Calling**. The concept is the same as OpenAI and Anthropic: define tools, let the model decide when to call them, execute the functions, and feed results back.

However, Google uses a different SDK structure — tools are defined as Python functions with type hints or as `genai.protos.FunctionDeclaration` objects.

In [ ]:
# ============================================================
# 🔧 Define Tools for Gemini
# ============================================================

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result.

    Args:
        expression: A math expression to evaluate, e.g. '675647 / 35'
    """
    allowed = set('0123456789+-*/.() ')
    if not all(c in allowed for c in expression):
        return "Error: Invalid characters"
    return str(round(eval(expression), 4))

def web_search(query: str) -> str:
    """Search the web for factual information.

    Args:
        query: The search query, e.g. 'population of Boston'
    """
    knowledge = {
        "population of boston": "Boston has a population of approximately 675,647 (2024).",
        "universities in boston": "Boston has approximately 35 colleges and universities.",
        "gdp of usa": "The US GDP in 2024 was approximately $28.78 trillion.",
        "largest companies": "The largest companies by market cap include Apple, Microsoft, NVIDIA, Amazon, and Alphabet.",
    }
    for key, val in knowledge.items():
        if key in query.lower():
            return val
    return f"No results for: {query}"

def string_analyzer(text: str, operation: str) -> str:
    """Analyze or transform a text string.

    Args:
        text: The text to analyze
        operation: One of: word_count, char_count, uppercase, reverse
    """
    ops = {
        "word_count": str(len(text.split())),
        "char_count": str(len(text)),
        "uppercase": text.upper(),
        "reverse": text[::-1],
    }
    return ops.get(operation, f"Unknown operation: {operation}")

# Gemini can auto-detect tool schemas from Python function signatures + docstrings!
tools = [calculator, web_search, string_analyzer]

print(f"✅ Defined {len(tools)} tools for Gemini")
print("💡 Gemini reads function docstrings to generate tool schemas automatically!")

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> Gemini's biggest advantage: it can <b>auto-generate tool schemas from Python functions</b>. You just pass the functions directly — no JSON Schema needed! The SDK reads function signatures, type hints, and docstrings. OpenAI and Anthropic both require manual JSON schemas.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Gemini Agent Loop</h2>
</div>

The Gemini agent loop uses `generate_content` with automatic function calling. Gemini can also handle function calls in a more integrated way.

In [ ]:
# ============================================================
# 🔄 Gemini Agent with Function Calling
# ============================================================

# Map function names to actual functions
tool_map = {
    "calculator": calculator,
    "web_search": web_search,
    "string_analyzer": string_analyzer,
}

def gemini_agent(question: str, max_steps: int = 5, verbose: bool = True) -> str:
    """An agent powered by Gemini with function calling."""

    model = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        tools=tools
    )
    chat = model.start_chat()

    if verbose:
        print(f"\n{'='*70}")
        print(f"🧑 Question: {question}")
        print(f"{'='*70}")

    response = chat.send_message(question)

    for step in range(1, max_steps + 1):
        # Check for function calls in the response
        function_calls = [part for part in response.parts if hasattr(part, 'function_call') and part.function_call.name]

        if not function_calls:
            # No function calls — we have the final answer
            final = response.text
            if verbose:
                print(f"\n🟢 Final Answer:")
                print(f"   {final}")
                print(f"{'='*70}")
            return final

        # Execute each function call
        responses = []
        for fc in function_calls:
            fn_name = fc.function_call.name
            fn_args = dict(fc.function_call.args)

            if verbose:
                print(f"\n🟢 Step {step} — Function Call")
                print(f"   🛠️  Tool: {fn_name}")
                print(f"   📥 Args: {fn_args}")

            result = tool_map.get(fn_name, lambda **_: "Unknown tool")(**fn_args)

            if verbose:
                print(f"   👁️  Result: {result}")

            responses.append(
                genai.protos.Part(
                    function_response=genai.protos.FunctionResponse(
                        name=fn_name,
                        response={"result": result}
                    )
                )
            )

        # Send function results back
        response = chat.send_message(responses)

    return "Agent exceeded max steps."

print("✅ Gemini agent ready!")

In [ ]:
# ============================================================
# 🧪 Test 1: Multi-Step Research
# ============================================================
gemini_agent("What is the population of Boston divided by the number of universities?")

In [ ]:
# ============================================================
# 🧪 Test 2: Multiple Tools
# ============================================================
gemini_agent(
    "Search for the GDP of the USA, calculate 5% of it, and tell me how many words are in your answer."
)

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ Three SDKs Compared</h2>
</div>

Let's compare the three major SDKs side by side.

In [ ]:
# ============================================================
# 📊 SDK Comparison Table
# ============================================================

comparison = {
    "Feature": ["Tool Schema", "Auto-schema", "Tool Call Signal", "Result Format", "Chat Memory", "Streaming", "Free Tier"],
    "OpenAI": [
        "{type:'function', function:{...}}",
        "No (manual JSON)",
        "message.tool_calls[]",
        "role: 'tool', tool_call_id",
        "Manual (pass messages)",
        "stream=True",
        "No (pay per token)"
    ],
    "Anthropic": [
        "{name, input_schema:{...}}",
        "No (manual JSON)",
        "stop_reason='tool_use'",
        "tool_result in user msg",
        "Manual (pass messages)",
        ".stream() context mgr",
        "No (pay per token)"
    ],
    "Google": [
        "Python functions directly",
        "Yes! (from docstrings)",
        "part.function_call",
        "FunctionResponse proto",
        "Built-in (start_chat)",
        "stream=True",
        "Yes (generous free tier)"
    ]
}

# Print as table
print("📊 SDK Comparison:\n")
print(f"{'Feature':25s} | {'OpenAI':30s} | {'Anthropic':30s} | {'Google':30s}")
print("-" * 120)
for i, feature in enumerate(comparison["Feature"]):
    print(f"{feature:25s} | {comparison['OpenAI'][i]:30s} | {comparison['Anthropic'][i]:30s} | {comparison['Google'][i]:30s}")

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Key Takeaway:</b> All three SDKs implement the same core pattern — <b>define tools, let model call them, feed results back</b>. Google's auto-schema from docstrings is the most ergonomic; OpenAI has the largest ecosystem; Anthropic has the most transparent reasoning. Knowing all three makes you <b>truly model-agnostic</b>.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "What is Google's unique advantage for defining tools compared to OpenAI and Anthropic?",
        "options": [
            "Google tools run faster because they use gRPC",
            "Google can auto-generate tool schemas from Python function signatures and docstrings",
            "Google tools don't need API keys",
            "Google tools can only be defined in YAML format"
        ],
        "answer": 1,
        "explanation": "Google's SDK can automatically generate tool schemas from Python functions by reading their type hints, parameter names, and docstrings. OpenAI and Anthropic both require you to manually write JSON Schema definitions."
    },
    {
        "q": "What is the common agent pattern shared by all three SDKs (OpenAI, Anthropic, Google)?",
        "options": [
            "Send prompt → get response → done (no loops)",
            "Define tools → model decides to call tools → execute functions → feed results back → repeat until done",
            "Train a custom model → deploy → serve predictions",
            "Upload data → fine-tune → evaluate → deploy"
        ],
        "answer": 1,
        "explanation": "All three SDKs share the same agent loop: (1) define tools with schemas, (2) send user message + tools to model, (3) model returns tool calls, (4) execute the functions locally, (5) feed results back, (6) repeat until the model returns a final text answer."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise: Build a Planning Agent

Build an agent that takes a complex task and breaks it into a step-by-step plan before executing. Replace each `-----` with the correct value.

In [ ]:
# ============================================================
# 🔧 Exercise: Planning Agent
# ============================================================
# Replace each ----- with the correct value

def create_plan(task: str) -> str:
    """Create a step-by-step execution plan for a complex task.

    Args:
        task: The task description to plan for
    """
    return f"Plan created for: {task}"

def execute_step(step_description: str) -> str:
    """Execute a single step from the plan.

    Args:
        step_description: What this step should accomplish
    """
    return f"Completed: {step_description}"

# Create a planning model
planning_model = genai.GenerativeModel(
    model_name="-----",              # Which Gemini model?
    tools=[create_plan, execute_step, ------, ------],  # Add calculator and web_search
    system_instruction="You are a planning agent. For complex tasks: first create a plan, then execute each step using available tools. Always plan before acting."
)

# Test the planning agent
chat = planning_model.------()       # How do you start a conversation?

task = "Research the GDP of USA, calculate what 1% of it is, and determine if that's more than Boston's population."
print(f"📋 Task: {task}\n")

response = chat.send_message(task)

# Agent loop
for step in range(10):
    fn_calls = [p for p in response.parts if hasattr(p, 'function_call') and p.function_call.name]
    if not fn_calls:
        print(f"\n🤖 Final: {response.text}")
        break

    results = []
    for fc in fn_calls:
        name = fc.function_call.name
        args = dict(fc.function_call.args)
        fn_map = {"create_plan": create_plan, "execute_step": execute_step, "calculator": calculator, "web_search": web_search}
        result = fn_map.get(name, lambda **_: "Unknown")(**args)
        print(f"  🛠️ {name}({list(args.values())[0][:50]}...) → {result[:80]}")
        results.append(genai.protos.Part(
            function_response=genai.protos.FunctionResponse(name=name, response={"result": result})
        ))
    response = chat.send_message(results)

**📝 Your Observations:** *(double-click to edit this cell)*

1. Did the agent create a plan before executing? How many steps? _[Your answer]_

2. How does Gemini's function calling ergonomics compare to OpenAI's? _[Your answer]_

3. Which SDK would you choose for your final project? Why? _[Your answer]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try these extensions on your own:</p>
  <ol style="font-size: 14px;">
    <li><b>Model-agnostic wrapper:</b> Build a Python class that wraps all 3 SDKs behind a single <code>agent.run(question)</code> interface</li>
    <li><b>Gemini grounding:</b> Explore Gemini's built-in Google Search grounding for real-time web data</li>
    <li><b>Cost comparison:</b> Run the same 10-question benchmark on all 3 models and compare token costs</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Workshop 2 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You built agents with Google's SDK and compared all three major AI platforms.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li>Google's SDK <b>auto-generates schemas</b> from Python functions (most ergonomic)</li>
      <li>All 3 SDKs share the <b>same agent loop pattern</b> (tools → call → result → repeat)</li>
      <li><b>OpenAI</b>: largest ecosystem | <b>Anthropic</b>: best reasoning | <b>Google</b>: best free tier + auto-schema</li>
      <li>Being <b>model-agnostic</b> is the most valuable skill in applied GenAI</li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Next →</b> M13: Workflow Automation with n8n</p>
</div>